# Calibration — Standards-Given Limits

A process behavior chart usually asks the data a question: *"given what I've seen, where should
the limits fall?"*  The limits are **estimated** from the data on hand — the center from the
grand mean, the spread from the average moving range.

Sometimes you want to turn that around. Once a process has been characterized — you *know* its
mean and its within-subgroup spread — you can **tell** the chart where the limits should be and
hold them fixed, instead of letting them drift as each new batch of data arrives. Wheeler calls
these *standards-given* limits; `processbehavior` calls the frozen `(mean, sigma)` pair a
**`Calibration`**.

This tutorial works a real before/after story on the bundled coffee-shop dataset:

1. formulate and inspect the study, then look at the ordinary data-derived chart;
2. quantify a process change with capability;
3. freeze the post-change "new normal" as a calibration and monitor against it;
4. add a second "pre-change" calibration to **prove** the improvement;
5. see how a calibration behaves across every chart type — and where it deliberately won't bend.

## The data

`load_coffee_shop()` returns a ready-to-formulate `ProcessBehavior`: a barista station's wait
time (seconds) measured four times a day, six days a week, across sixteen weeks. A new espresso
machine goes in at the **start of week 8 (2026-02-23)**.

We make **`week` the rational subgroup** and **`date` the time axis** — `formulate` needs a
factor × time structure, and grouping by week lets the daily readings form rational subgroups
while the date axis keeps the series in calendar order.

In [ ]:
from processbehavior import Calibration, load_coffee_shop

pb = load_coffee_shop()
study = pb.formulate(response='wait_sec', time='date', factors=['week'])
print(study)

## Inspect the formulation

Before charting, see what the study supports. `formulate()` detects the design-state lineage
(planned → observed → **analytical**), and the analytical state (ADS) decides which charts are
valid and how variation is decomposed. This is a complete, well-formed `Study`; calibration is
just one more thing we can ask of it.

In [ ]:
print('analytical design state (ADS):', study.analytical_design_state.sds)
print('valid charts :', study.valid_charts)
print('recommended  :', study.recommended_chart)

## The ordinary, data-derived chart

Here is the individuals (X) chart for the whole series — limits estimated from the data, the
default behavior. Watch what the espresso-machine change does to it.

In [ ]:
study.execute(chart='X', by=[]).plot()

The center line lands around **219 s** — halfway between the two eras, a wait time the process
never actually held. Worse, the limits are wide: the moving range is inflated by the week-7→8
*step*, so the estimated spread (overall σ ≈ 40 s) is roughly double the real within-week spread
(≈ 20 s). **The data-derived limits absorb the change instead of flagging it.** That is exactly
the situation calibration is built for.

## Quantify the change with capability

How big was the change? Capability views over a time **window** answer that. The window is
half-open on the study's time axis (here, `date`), so the change date `2026-02-23` cleanly
splits the eras:

- `window=(None, CHANGE)` → **weeks 1–7**, the old process;
- `window=(CHANGE, None)` → **weeks 8–16**, the new normal.

In [ ]:
CHANGE = '2026-02-23'

before = study.capability(usl=240, target=180, window=(None, CHANGE))
after  = study.capability(usl=240, target=180, window=(CHANGE, None))
full   = study.capability(usl=240, target=180)

print(f'before  (weeks 1-7) : n={before.n:3d}  mean={before.y_bar:6.1f}s  Ppk={before.ppk:.2f}')
print(f'after (weeks 8-16)  : n={after.n:3d}  mean={after.y_bar:6.1f}s  Ppk={after.ppk:.2f}')
print(f'within-process sigma (pooled R2): {full.sigma_hat_r2:.1f}s')

The mean dropped from ≈ **240 s** to ≈ **203 s**, and the pooled within-subgroup σ — the spread
of *individual* readings around their own subgroup — is about **20 s**. Those are the numbers we
freeze into calibrations. (We round to tidy "house standard" values; an analyst would set these
from process knowledge, not to three decimals.)

## Calibration #1 — freeze the "new normal"

A `Calibration` is a `(label, mean, sigma)` value object. `sigma` is the within-subgroup
standard deviation of **individual** values — used *as-is*, applied forward to place the limits;
it is never run back through estimator constants to "recover" a process sigma.

Apply it at execute time with `calibration=`. Now the limits say "here is where a process
centered at 203 s with σ = 20 s lives," regardless of what this particular data does.

In [ ]:
post = Calibration(label='post-change normal', mean=203.0, sigma=20.0)

study.execute(chart='X', by=[], calibration=post).plot()

Against the frozen new standard (203 ± 3·20 = **[143, 263]**), the verdict is blunt:

- every reading in **weeks 1–7** sits **above** the upper limit — the old process was nowhere
  near the new normal;
- the **week-12 spike back to ~237 s** trips the limit too — a transient regression the
  data-derived chart (which had swallowed the whole series) quietly missed.

The limits no longer breathe with the data; they hold the line at the standard.

## Calibration #2 and the named set — prove the improvement

Flip the question: against the **old** baseline, did the change actually help? Build a second
calibration for the pre-change era, then attach both to the study by label with
`with_calibration()`. It returns a *new* study (immutable — the original is untouched), and you
select a calibration at execute time by its label.

In [ ]:
pre = Calibration(label='pre-change baseline', mean=240.0, sigma=20.0)

monitored = study.with_calibration(pre).with_calibration(post)
print('attached labels :', sorted(monitored.calibrations))
print('original study  :', dict(study.calibrations), '(unchanged)')

monitored.execute(chart='X', by=[], calibration='pre-change baseline').plot()

Against the old baseline (240 ± 60 = **[180, 300]**), the week-8-onward readings fall **below**
the lower limit — a run of points beyond 3σ on the good side. That is not noise; it is the
process genuinely operating at a new, lower level. The improvement is real, and a frozen
baseline is what lets you say so with a control chart.

## One calibration, every chart type

The same `Calibration` works on any chart — but *how* the sigma lands depends on what the chart
plots. There are exactly two families:

| Chart | Center | Limits | Why |
|---|---|---|---|
| **X** (individuals) | `mean` | `mean ± 3σ` | plots individuals → σ direct |
| **Xbar** (subgroup mean) | `mean` | `mean ± n_sigma·σ/√N` | means vary less than individuals → `√N` |
| **S** (subgroup spread) | `c4(N)·σ` | `B5(N)·σ … B6(N)·σ` | plots a *dispersion statistic* → constants |
| **mR** (moving range) | `d2·σ` | `0 … D4·d2·σ` | dispersion statistic → constants |

`σ` is always the **individual** standard deviation you supplied. Location charts express the
limits constant-free in σ; dispersion charts (S, mR) necessarily carry their
sampling-distribution constants because their y-axis *is* a spread, not σ. So a calibrated Xbar
band is narrower than σ by `√N`, and a calibrated S centers *below* the σ you typed — both
correct, neither a bug.

In [ ]:
xbar = monitored.execute(chart='Xbar', calibration='post-change normal')
print('Xbar :', xbar.get_statistics('Xbar'))   # center 203, half-width 3*20/sqrt(N)

s_chart = monitored.execute(chart='S', calibration='post-change normal')
print('S    :', s_chart.get_statistics('S'))    # center c4(N)*20, asymmetric band

companion = monitored.execute(chart='X', by=[], companion=True, calibration='post-change normal')
print('mR   :', companion.get_statistics('mR')) # center d2*20, UCL D4*d2*20
companion.plot()

## Residuals, too

Calibration works on residual charts as well. A **plain** residual is already centered at zero,
so the calibration mean is ignored and the limits are simply `0 ± n_sigma·σ` — useful for asking
"is the *within-subgroup* noise consistent with my standard σ?" (Use `recentered=True` to shift a
residual back onto the response scale; it then centers at `calibration.mean`.)

In [ ]:
resid = monitored.execute(chart='X', value='R2', by=[], calibration='post-change normal')
print('R2 (within-subgroup residual):', resid.get_statistics('X'))  # center 0, +/- 3*sigma

## `n_sigma` and the 3-sigma rule

`n_sigma` (the limit width multiplier) and the calibration are independent knobs: the calibration
sets the center and σ, `n_sigma` sets the width. They **compose** on Xbar/S. But the individuals
(X) and moving-range (mR) charts are fixed at 3-sigma, and a calibration is *not* a back door to
a non-3-sigma individuals limit — so a calibrated X/mR rejects a non-default `n_sigma`.

In [ ]:
# Composes on Xbar:
wide = monitored.execute(chart='Xbar', calibration='post-change normal', n_sigma=2.0)
print('Xbar @ 2-sigma:', wide.get_statistics('Xbar'))

# Rejected on X:
try:
    monitored.execute(chart='X', by=[], calibration='post-change normal', n_sigma=2.5)
except Exception as e:
    print(f'{type(e).__name__}: {e}')

## When to calibrate

Reach for a calibration when you have a **known, trustworthy** mean and σ and want the chart to
hold them — rather than re-estimate from whatever data is in front of it:

- **Monitor forward.** Once a process is characterized and stable, freeze its limits so they
  don't drift with new noise; new data is judged against the known-good state.
- **Audit a change.** Compare a new era against an old, frozen baseline (as we did above) to
  decide whether a shift is real.
- **Apply a spec-driven or engineering standard** that the data should conform to.

The default (`calibration=None`) is unchanged — limits are estimated from the data exactly as
before. When a calibration *is* in force, the result records it: each chart's metadata carries
`limits_source='calibration'` along with the label, mean, and sigma used.